# Parameter setzen:

In [ ]:
import os
import numpy as np
import sys
import scipy.io as sio
from pathlib import Path
%matplotlib widget

# GPU wählen
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Run-Config
model = "Proton_After_WALINET_3DMask_tMPPCA"
batch_size = 20
Vol = "5_Brisbane"#_Brisbane"
add_water = True # adde water back grom hankel decomposition

# 🔹 NEU: Ordner + Dateiname getrennt
data_dir = Path(f"../datasets/Proton/B0corrected_wo_LipidMask/Vol{Vol}/OriginalData")
file_name = "data_after_walinet_tMPPCA.npy"

model_input = data_dir / file_name

# ---- Comparisons definieren ----
comparison_paths = [
    model_input,
    # GT_path,
]

Titles = [
    "Deep",
    "No Denoising",
    # "GT",
]

# ---- Daten laden (einmalig!) ----
Data = []

for p in comparison_paths:
    p = Path(p)

    if p.suffix == ".npy":
        arr = np.load(p)

    elif p.suffix == ".mat":
        mat = loadmat(p)
        arr = np.asarray(mat["csi"]["Data"][0,0])

    else:
        raise ValueError(f"Unsupported file format: {p}")

    Data.append(arr)

# Inference

In [ ]:
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

from denoising.config.load import load_yaml
from denoising.config.build import build_config
from denoising.inference.api import infer
from src.denoising.figures.EvalBeforeFitting import *

config_path = f"../trained_models/{model}/train.yaml"

cfg = build_config(load_yaml(config_path))

denoised, meta = infer(
    cfg=cfg,
    ckpt_path=f"../trained_models/{model}/checkpoints/last.pt",
    input_path=model_input,   # exakter Pfad zur Datei
    batch_size=batch_size
)

Data.insert(0, denoised)

# Optional: Wasser auf alle Datensätze addieren
if add_water:
    raw_data = np.load(data_dir / "data.npy")
    suppressed_water = np.load(data_dir / "SupressedWater.npy")
    water = raw_data - suppressed_water

    Data = [d + water for d in Data]

# auch fft
Data_ft = [np.fft.fftshift(np.fft.fft(d, axis=3), axes=3) for d in Data]

In [ ]:
# Data_denoised = np.swapaxes(Data[0], 0, 1)
# #Data_noisy = np.swapaxes(Data[1], 0, 1)

# sio.savemat(f'Vol{Vol}_Denoised_after_Walinet_3DMask_tMPPCA.mat', {'Data': Data_denoised})
# #sio.savemat(f'Vol{Vol}_Noisy_B0_correction_after_Walinet_tMPPCA.mat', {'Data': Data_noisy})

# Optional als matlab datei speichern

In [ ]:
# import numpy as np 

# out_data = np.load("data_denoised.npy")

# from scipy.io import loadmat, savemat

# savemat('sf_brain_DMI_HC_pilot_Combined_Model.mat', {'Data': out_data})

# # Für SIMS:

# ### CAREFUL DISTINGUISH BETWEEN LESION AND NO LESION

# #/Lade Par und mask aus der B0_1.mat-Datei
# b0_data = loadmat('../datasets/Simulated_Lesion_GT/Lesion_120pts.mat')  #Simulated_Lesion_GT/Lesion_120pts.mat  Simulated_ground_truth/B0_1.mat
# par = b0_data['csi_data_lr']['Par'][0, 0]
# mask = b0_data['csi_data_lr']['mask'][0, 0]

# # Deine Datensätze
# data_dicts = {
#     f'Lesion_double_deep_tmppca_6_retrained.mat': denoised
#     #f'Lesion_double_deep_noisy_6_uncorrelated.mat': noisy_data
#     #'Lesion_gt.mat': tgt_data
# }

# # Für jede Datei speichern
# for filename, data in data_dicts.items():
#     savemat(filename, {
#         'csi_data_lr': {
#             'Data': data,
#             'Par': par,
#             'mask': mask
#         }
#     })

# Compare spectral slice

In [ ]:
# fig, axes = plot_z_slices(
#     Data_ft,
#     Titles,
#     t=700,
#     T=7,
#     share_clim="global",
# )

# plt.show()

# Compare Spectra

In [ ]:
# _ = plot_voxel_spectra_over_z(Data_ft, Titles, x=29, y=30, min_t_index=500, mag=False)#, z_max=30)#, min_t_index=500, max_t_index=600)

# plt.show()

# Compare average spectra
Here I compare the average spectrum over time (which is a high SNR estimate)

In [ ]:
Avg = [np.mean(d, axis=(0,1,2)) for d in Data_ft]

_ = plot_average_spectra_over_T(
    Avg,
    Titles,
    n_cols=2,
)
plt.show()

# Compare spectra interactively

In [ ]:
data1 = np.real(Data_ft[0])
data2 = np.real(Data_ft[1])

image_volume = np.sum((np.real(Data[0])), axis=-1)   # shape (X, Y, Z)

interactive_spectra_viewer(
    image_volume=image_volume,
    spectrum_func1=lambda x, y, z: data1[x, y, z, :],
    spectrum_func2=lambda x, y, z: data2[x, y, z, :],
    label1="Deep",
    label2="No Denoising",
    f_min=500,
    #f_max=60,
    cmap="plasma",
)